# 🏛️ Cuaderno 1 de 2 · M1 + M2 (preprocesar y HyperNCD)

**Qué hace:** toma la escena etiquetada (`listo.zip`), la une, corre el **Módulo 1** (preprocesamiento) y el **Módulo 2** (HyperNCD, `f_sem`). Todo lo que calcula lo **deja en tu Drive**.
**Después:** abre el **cuaderno 2** (`colab_2_m3_m4.ipynb`), que lee de ahí y corre M3 y M4.

> ⚠️ **GPU:** usa **la GPU con la que ya te funcionó M2 antes** (con `colab_runner.ipynb`). Si no la recuerdas, prueba **T4**. M2 usa MinkowskiEngine, que no funciona en todas las GPU. Si da `CUDA error ... concurrent_unordered_map`, cambia a **A100** y vuelve a correr desde la celda 1.

**Antes de empezar**
- `listo.zip` en tu Drive: `PFC1_HyperNCD_Sonata/m1/entrada/etiquetado/listo.zip` (la celda 4 te deja subirlo si falta).
- El checkpoint de HyperNCD ya está en `m2/entrada/` (el de la época 8).

**Corre las celdas en orden, de la 1 a la 8.** Si se cae la sesión, vuelve a correr desde la 1: lo ya guardado en Drive no se repite (`REHACER = True` lo fuerza).

| Celda | Qué hace |
|---|---|
| 1-2 | GPU, Drive, parámetros y repo |
| 3 | instala `laspy` y `open3d` (lo que necesita M1) |
| 4 | une las 19 piezas y guarda las etiquetas |
| 5 | **M1**: SOR, voxel de 5 cm y normales; pasa las etiquetas a los puntos |
| 6-7 | **M2**: restaura el entorno de HyperNCD de Drive y calcula `f_sem` |
| 8 | comprueba que todo quedó en Drive |

In [ ]:
# @title 1. GPU, Drive, carpetas y PARÁMETROS
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import os
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, glob

DRIVE = "/content/drive/MyDrive/PFC1_HyperNCD_Sonata"
M1_ENT, M1_SAL = f"{DRIVE}/m1/entrada", f"{DRIVE}/m1/salida"
M2_ENT, M2_SAL = f"{DRIVE}/m2/entrada", f"{DRIVE}/m2/salida"
M3_ENTRENA, M3_PESOS, M3_SAL = f"{DRIVE}/m3/entrenamiento", f"{DRIVE}/m3/pesos", f"{DRIVE}/m3/salida"
M4_SAL   = f"{DRIVE}/m4/salida"
ENTORNOS = f"{DRIVE}/entornos"
for _d in (M1_ENT, M1_SAL, M2_ENT, M2_SAL, M3_ENTRENA, M3_PESOS, M3_SAL, M4_SAL, ENTORNOS):
    os.makedirs(_d, exist_ok=True)

def _mover(origen, destino):
    """Mueve origen -> destino solo si origen existe. Nunca pisa nada: si destino existe, no lo toca."""
    if not os.path.exists(origen):
        return
    if os.path.exists(destino):
        if os.path.isdir(origen) and os.path.isdir(destino):   # fusiona carpetas
            for x in os.listdir(origen):
                _mover(os.path.join(origen, x), os.path.join(destino, x))
            try:
                os.rmdir(origen)
            except OSError:
                pass
        else:
            print("  ya existe, no se toca:", destino.replace(DRIVE + "/", ""))
        return
    os.makedirs(os.path.dirname(destino), exist_ok=True)
    shutil.move(origen, destino)
    print("  movido:", origen.replace(DRIVE + "/", ""), "->", destino.replace(DRIVE + "/", ""))

# M1: la salida (los .npy preprocesados, empaquetados)
_mover(f"{DRIVE}/data_m1.tar.gz",    f"{M1_SAL}/data_m1.tar.gz")
_mover(f"{DRIVE}/m1/data_m1.tar.gz", f"{M1_SAL}/data_m1.tar.gz")
# M2: checkpoints de entrada y resultados
_mover(f"{DRIVE}/datos_m2_descomprimidos",         f"{M2_ENT}/datos_m2_descomprimidos")
_mover(f"{DRIVE}/resultados_m2",                   f"{M2_SAL}/resultados_m2")
_mover(f"{DRIVE}/resultados_modulo2_hyperncd.zip", f"{M2_SAL}/resultados_modulo2_hyperncd.zip")
# Entornos compilados
_mover(f"{DRIVE}/hyperncd_env_cache.tar.gz", f"{ENTORNOS}/hyperncd_env_cache.tar.gz")
for _p in glob.glob(f"{DRIVE}/pointcept_env_*.tar.gz"):
    _mover(_p, f"{ENTORNOS}/{os.path.basename(_p)}")
# M3: si ya existía la carpeta anterior de la adaptación
_mover(f"{DRIVE}/sonata_adaptado/entrenamiento",       M3_ENTRENA)
_mover(f"{DRIVE}/sonata_adaptado/sonata_catedral.pth", f"{M3_PESOS}/sonata_catedral.pth")
_mover(f"{DRIVE}/sonata_adaptado/features",            M3_SAL)
print("Estructura lista en", DRIVE)


# ===================== PARÁMETROS (edita aquí) =====================
ESCENA = "capilla_etiquetada"                                  # nombre de la escena (carpetas y archivos en Drive)
ZIP_ETIQUETADO = f"{M1_ENT}/etiquetado/listo.zip"              # zip con un .las por pieza (la clase es el nombre del archivo)
CKPT_M2 = "AUTO"                                               # "AUTO" = HyperNCD split0_run4 época 8; o la ruta a otro .ckpt
REHACER = False                                                # True = ignora lo guardado en Drive y recalcula
BRANCH = "main"
# ====================================================================
WORK = "/content/work"; os.makedirs(WORK, exist_ok=True)
LOGS = f"{DRIVE}/m4/salida/{ESCENA}/logs"; os.makedirs(LOGS, exist_ok=True)   # lo que imprime cada módulo queda aquí

In [ ]:
# @title 2. Clonar / actualizar tu repo (rama indicada en BRANCH)
import os
from getpass import getpass

REPO_OWNER, REPO_NAME = "manyorycuevamendoza", "PFC1-HyperNCD-Sonata"
REPO_DIR = f"/content/{REPO_NAME}"

token = None
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
    if token:
        print("✅ Token leído desde Colab Secrets (GITHUB_TOKEN).")
except Exception:
    pass
if not token:
    token = getpass("Ingresa tu GitHub Personal Access Token (oculto): ").strip()

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} https://{token}@github.com/{REPO_OWNER}/{REPO_NAME}.git {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull origin {BRANCH}
%cd {REPO_DIR}
!git log -1 --oneline
for f in ("experimentacion/evaluacion_etiquetada/preparar_etiquetas.py", "preprocesar.py", "predecir_hyperncd.py"):
    assert os.path.exists(f"{REPO_DIR}/{f}"), f"Falta {f}: haz commit y push de ese archivo a la rama {BRANCH}."

In [ ]:
# @title 3. Dependencias de M1 (laspy y open3d)
!pip install -q laspy open3d
import sys
try:
    import laspy, open3d
    print("✅ laspy", laspy.__version__, "| open3d", open3d.__version__)
except Exception as e:
    raise RuntimeError(f"No se pudo importar laspy/open3d en este Python ({sys.version.split()[0]}): {e}")
PYM1 = sys.executable          # M1 corre con el Python de Colab

In [ ]:
# @title 4. Datos etiquetados: descomprimir el zip, unir las piezas y guardar las etiquetas
import os, glob, shutil, zipfile

RAW = "/content/etiquetado_raw"
CACHE_ENT = f"{M1_ENT}/{ESCENA}"; os.makedirs(CACHE_ENT, exist_ok=True)
MERGED_LAS = f"{WORK}/{ESCENA}.las"
ETIQ_NPZ = f"{WORK}/{ESCENA}_etiquetas.npz"
c_las, c_npz = f"{CACHE_ENT}/{ESCENA}.las", f"{CACHE_ENT}/{ESCENA}_etiquetas.npz"

if os.path.exists(c_las) and os.path.exists(c_npz) and not REHACER:
    print("⚡ Escena ya unida en Drive; se copia.")
    shutil.copy2(c_las, MERGED_LAS); shutil.copy2(c_npz, ETIQ_NPZ)
else:
    if not os.path.exists(ZIP_ETIQUETADO):
        print("No encuentro el zip en Drive. Súbelo ahora (queda guardado para la próxima):")
        from google.colab import files
        sub = files.upload()
        os.makedirs(os.path.dirname(ZIP_ETIQUETADO), exist_ok=True)
        shutil.move(next(iter(sub)), ZIP_ETIQUETADO)
    shutil.rmtree(RAW, ignore_errors=True); os.makedirs(RAW)
    with zipfile.ZipFile(ZIP_ETIQUETADO) as zf:
        for m in zf.namelist():
            destino = os.path.realpath(os.path.join(RAW, m))
            assert destino.startswith(os.path.realpath(RAW)), f"ruta sospechosa en el zip: {m}"
        zf.extractall(RAW)
    print("Archivos .las:", len(glob.glob(f"{RAW}/**/*.las", recursive=True)))
    !{PYM1} {REPO_DIR}/experimentacion/evaluacion_etiquetada/preparar_etiquetas.py unir --carpeta {RAW} --salida-las {MERGED_LAS} --salida-etiq {ETIQ_NPZ}
    shutil.copy2(MERGED_LAS, c_las); shutil.copy2(ETIQ_NPZ, c_npz)

In [ ]:
# @title 5. M1 · Preprocesar la escena (SOR + voxel 5 cm + normales) y pasar las etiquetas a los puntos
import os, shutil

NPY = f"{WORK}/{ESCENA}.npy"
CACHE_M1 = f"{M1_SAL}/{ESCENA}"; os.makedirs(CACHE_M1, exist_ok=True)
archivos = [f"{ESCENA}.npy", f"{ESCENA}.json", f"{ESCENA}_etiquetas.npy", f"{ESCENA}_etiquetas.json"]
if all(os.path.exists(f"{CACHE_M1}/{a}") for a in archivos) and not REHACER:
    print("⚡ M1 ya calculado en Drive; se copia.")
    for a in archivos: shutil.copy2(f"{CACHE_M1}/{a}", f"{WORK}/{a}")
else:
    !{PYM1} {REPO_DIR}/preprocesar.py --input {MERGED_LAS} --output {WORK}/{ESCENA}
    !{PYM1} {REPO_DIR}/experimentacion/evaluacion_etiquetada/preparar_etiquetas.py mapear --npy {NPY} --json {WORK}/{ESCENA}.json --etiquetas {ETIQ_NPZ}
    for a in archivos: shutil.copy2(f"{WORK}/{a}", f"{CACHE_M1}/{a}")
import numpy as np
S = np.load(NPY, mmap_mode="r"); print("M1 listo:", S.shape, "->", CACHE_M1)

## M2 · HyperNCD (entorno aparte: Python 3.8 + MinkowskiEngine)
La celda 6 es la misma que usas en `colab_runner.ipynb`: restaura `entornos/hyperncd_env_cache.tar.gz` de Drive (~20 s) o lo compila la primera vez.

In [ ]:
# @title 6. M2 · Instalar o Restaurar hyperncd-env + MinkowskiEngine (Guarda caché en Drive como imagen lista)
%%bash
set -e

ENV_CACHE="/content/drive/MyDrive/PFC1_HyperNCD_Sonata/entornos/hyperncd_env_cache.tar.gz"

if [ ! -d "/content/miniconda3" ]; then
    echo "[1/4] Instalando Miniconda3 base..."
    wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
    bash /tmp/miniconda.sh -b -p /content/miniconda3
fi

source /content/miniconda3/etc/profile.d/conda.sh
conda config --set always_yes yes
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main 2>/dev/null || true
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r 2>/dev/null || true

# Si ya existe la imagen del entorno compilada en tu Google Drive, la restauramos en ~20 segundos
if [ -f "$ENV_CACHE" ] && [ ! -d "/content/miniconda3/envs/hyperncd-env" ]; then
    echo "⚡ Encontrado entorno pre-compilado en tu Drive ($ENV_CACHE). Restaurando en ~20 segundos..."
    mkdir -p /content/miniconda3/envs/hyperncd-env
    tar -xzf "$ENV_CACHE" -C /content/miniconda3/envs/hyperncd-env
    /content/miniconda3/envs/hyperncd-env/bin/python -c "import torch, MinkowskiEngine as ME; print('✅ Entorno restaurado desde Drive en segundos! MinkowskiEngine:', ME.__version__)"
    exit 0
fi

if [ ! -d "/content/miniconda3/envs/hyperncd-env" ]; then
    echo "[2/4] Creando entorno hyperncd-env (Python 3.8 + CUDA 11.8 + OpenBLAS)..."
    conda create -y -n hyperncd-env python=3.8 -c conda-forge
    conda install -y -n hyperncd-env -c nvidia/label/cuda-11.8.0 cuda-nvcc cuda-cudart-dev cuda-libraries-dev
    conda install -y -n hyperncd-env -c conda-forge openblas-devel ninja gxx_linux-64=11
fi

conda activate hyperncd-env

echo "[3/4] Instalando PyTorch 1.10.1+cu113 y dependencias de HyperNCD..."
pip install -q torch==1.10.1+cu113 torchvision==0.11.2+cu113 -f https://download.pytorch.org/whl/torch_stable.html
pip install -q pytorch-lightning==1.4.8 torchmetrics==0.7.2 scipy==1.7.3 pyyaml tqdm scikit-learn plotly matplotlib
pip install -q torch-scatter -f https://data.pyg.org/whl/torch-1.10.0+cu113.html

if ! python -c "import MinkowskiEngine" 2>/dev/null; then
    echo "[4/4] Compilando e instalando MinkowskiEngine (solo esta primera vez)..."
    # Asegurar cabecera nvtx3/nvToolsExt.h que pide MinkowskiEngine
    conda install -y -n hyperncd-env -c nvidia/label/cuda-11.8.0 cuda-nvtx-dev 2>/dev/null || true
    if [ ! -f "$CONDA_PREFIX/include/nvtx3/nvToolsExt.h" ]; then
        git clone -q --depth 1 https://github.com/NVIDIA/NVTX.git /tmp/NVTX
        cp -r /tmp/NVTX/c/include/nvtx3 "$CONDA_PREFIX/include/"
        rm -rf /tmp/NVTX
    fi
    if [ ! -d "/content/MinkowskiEngine" ]; then
        git clone -q https://github.com/NVIDIA/MinkowskiEngine.git /content/MinkowskiEngine
    fi
    cd /content/MinkowskiEngine
    export CUDA_HOME="$CONDA_PREFIX"
    export PATH="$CONDA_PREFIX/bin:$PATH"
    export CPLUS_INCLUDE_PATH="$CONDA_PREFIX/include:${CPLUS_INCLUDE_PATH:-}"
    export TORCH_CUDA_ARCH_LIST="7.5;8.0;8.6"
    export MAX_JOBS=4
    export CC="$CONDA_PREFIX/bin/x86_64-conda-linux-gnu-gcc"
    export CXX="$CONDA_PREFIX/bin/x86_64-conda-linux-gnu-g++"
    python setup.py install --blas_include_dirs=${CONDA_PREFIX}/include --blas=openblas
fi

python -c "import torch, MinkowskiEngine as ME; print('✅ MinkowskiEngine listo, versión:', ME.__version__, '| GPU:', torch.cuda.get_device_name(0))"

# Guardar imagen del entorno ya compilado en tu Google Drive para no volver a compilar nunca más
if [ -d "/content/drive/MyDrive" ] && [ ! -f "$ENV_CACHE" ]; then
    echo "💾 Guardando imagen del entorno compilado en tu Drive ($ENV_CACHE) para que la próxima vez cargue en 20 segundos..."
    mkdir -p "$(dirname "$ENV_CACHE")"
    tar -czf "$ENV_CACHE" -C /content/miniconda3/envs/hyperncd-env .
    echo "✅ Imagen del entorno guardada en Drive."
fi

In [ ]:
# @title 7. M2 · f_sem de la escena con HyperNCD
import glob, os, re, shutil, subprocess

PY_M2 = "/content/miniconda3/envs/hyperncd-env/bin/python"
FSEM = f"{WORK}/{ESCENA}_fsem.npz"
C_FSEM = f"{M2_SAL}/{ESCENA}/{ESCENA}_fsem.npz"; os.makedirs(os.path.dirname(C_FSEM), exist_ok=True)
C_CKPT = C_FSEM.replace(".npz", ".ckpt.txt")          # recuerda con qué checkpoint se calculó

ES_AUTO = (CKPT_M2 == "AUTO")
if ES_AUTO:
    # El correcto es UNO solo: HyperNCD split0_run4, mejor época (8, mIoU de validación 0,384). Si no está, se detiene:
    # nunca elige "otro parecido" (antes tomaba split0_full epoch=0, casi sin entrenar).
    ESPERADO = "SemanticPOSS_S0_best-epoch=8-step=11196.ckpt"
    ck = glob.glob(f"{M2_ENT}/**/*.ckpt", recursive=True)
    exactos = [c for c in ck if os.path.basename(c) == ESPERADO and "split0_run4" in c]
    assert exactos, (f"No encuentro {ESPERADO} dentro de split0_run4 en {M2_ENT}.\nCheckpoints que hay:\n  " +
                     "\n  ".join(sorted(ck)) + "\nSi quieres otro, pon su ruta en CKPT_M2.")
    CKPT_M2 = exactos[0]
print("Checkpoint de M2:", CKPT_M2)

mismo_ckpt = os.path.exists(C_CKPT) and open(C_CKPT).read().strip() == CKPT_M2
if os.path.exists(C_FSEM) and mismo_ckpt and not REHACER:
    print("⚡ M2 ya calculado con este checkpoint; se copia desde Drive.")
    shutil.copy2(C_FSEM, FSEM)
else:
    r = subprocess.run([PY_M2, f"{REPO_DIR}/predecir_hyperncd.py", "--input", NPY, "--ckpt", CKPT_M2, "--output", FSEM],
                       cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(r.stdout[-4000:]); open(f"{LOGS}/m2.log", "w").write(r.stdout)
    assert r.returncode == 0, "Falló M2 (mira el error de arriba; si dice CUDA error, cambia de GPU: ver la nota del inicio)"
    for ext in (".npz", ".json", ".csv"):
        shutil.copy2(FSEM.replace(".npz", ext), C_FSEM.replace(".npz", ext))
    open(C_CKPT, "w").write(CKPT_M2)
    import json
    info = json.load(open(FSEM.replace(".npz", ".json")))["ckpt_info"]
    print(f"Checkpoint realmente cargado: época {info['epoch']}, paso {info['global_step']}")
    if ES_AUTO:
        assert info["epoch"] == 8, f"Se cargó la época {info['epoch']} y se esperaba la 8: revisa el checkpoint"

In [ ]:
# @title 8. Comprobar que M1 y M2 quedaron en Drive (lo que lee el cuaderno 2)
import os
esperados = [f"{M1_SAL}/{ESCENA}/{ESCENA}.npy", f"{M1_SAL}/{ESCENA}/{ESCENA}.json",
             f"{M1_SAL}/{ESCENA}/{ESCENA}_etiquetas.npy", f"{M1_SAL}/{ESCENA}/{ESCENA}_etiquetas.json",
             f"{M2_SAL}/{ESCENA}/{ESCENA}_fsem.npz"]
faltan = [p for p in esperados if not os.path.exists(p)]
for p in esperados:
    print("✅" if os.path.exists(p) else "❌", p.replace(DRIVE + "/", ""),
          f"({os.path.getsize(p)/2**20:.1f} MB)" if os.path.exists(p) else "")
assert not faltan, "Faltan archivos: revisa la celda que los genera"
print("\nListo. Ahora abre el cuaderno 2 (colab_2_m3_m4.ipynb): puede ser otra GPU, por ejemplo la L4.")